In [ ]:
!pip install -q audiomentations timm kagglehub

In [5]:
import os
import gc
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import timm

from tqdm import tqdm

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

from torch.utils.data import Dataset, DataLoader

from audiomentations import (
    Compose,
    AddGaussianNoise,
    PitchShift,
    TimeStretch
)

In [ ]:
import kagglehub

os.environ["KAGGLE_API_TOKEN"] = os.getenv("KAGGLE_API_TOKEN")

path = kagglehub.competition_download(
    'itmo-acoustic-event-detectin-2026'
)

print("Path:", path)

competition_path = path

train_folder = os.path.join(
    competition_path,
    "audio_train",
    "train"
)

train_csv = os.path.join(
    competition_path,
    "train.csv"
)

print("Files:", len(os.listdir(train_folder)))

100%|██████████| 7.34G/7.34G [01:37<00:00, 80.8MB/s]

Extracting files...


Path: /root/.cache/kagglehub/competitions/itmo-acoustic-event-detectin-2026
Files: 5683


In [19]:
SEED = 42

TARGET_SR = 32000
MAX_LEN = TARGET_SR * 5

BATCH_SIZE = 24
EPOCHS = 12

N_FOLDS = 5

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(DEVICE)

SAVE_DIR = "/content"
os.makedirs(SAVE_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

cuda


In [9]:
df = pd.read_csv(
    train_csv,
    skiprows=1,
    names=["fname", "label"]
)

df["path"] = df["fname"].apply(
    lambda x: os.path.join(train_folder, x)
)

labels = sorted(df["label"].unique())

label2id = {
    l: i for i, l in enumerate(labels)
}

id2label = {
    i: l for l, i in label2id.items()
}

df["label_id"] = df["label"].map(label2id)

print("Classes:", len(labels))

Classes: 41


In [10]:
augment = Compose([

    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.5),
    TimeStretch(min_rate=0.9, max_rate=1.1, p=0.3),
    PitchShift(min_semitones=-2, max_semitones=2, p=0.3),
    ])

In [11]:
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=TARGET_SR,
    n_fft=2048,
    hop_length=512,
    n_mels=128
)

time_mask = torchaudio.transforms.TimeMasking(
    time_mask_param=30
)

freq_mask = torchaudio.transforms.FrequencyMasking(
    freq_mask_param=15
)

def load_audio(path, is_train=True):

    wav, sr = torchaudio.load(path)

    wav = wav.mean(dim=0)

    if sr != TARGET_SR:

        wav = torchaudio.functional.resample(
            wav,
            sr,
            TARGET_SR
        )

    wav = wav.numpy()

    if is_train:
        wav = augment(
            samples=wav,
            sample_rate=TARGET_SR
        )

    if len(wav) <= MAX_LEN:

        wav = np.pad(
            wav,
            (0, MAX_LEN - len(wav))
        )

    else:

        max_start = len(wav) - MAX_LEN

        start = np.random.randint(
            0,
            max_start + 1
        )

        wav = wav[
            start:start + MAX_LEN
        ]

    wav = torch.tensor(
        wav,
        dtype=torch.float32
    )

    mel = mel_transform(wav)

    mel = torch.log(mel + 1e-6)

    if is_train:

        mel = time_mask(mel)
        mel = freq_mask(mel)

    delta = torchaudio.functional.compute_deltas(mel)

    delta2 = torchaudio.functional.compute_deltas(delta)

    mel = torch.stack([
        mel,
        delta,
        delta2
    ])

    return mel

In [12]:
class AudioDataset(Dataset):

    def __init__(self, df, train=True):

        self.df = df.reset_index(drop=True)
        self.train = train

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        x = load_audio(
            row["path"],
            is_train=self.train
        )

        y = row["label_id"]

        return x, y

In [13]:
class AudioModel(nn.Module):

    def __init__(self, n_classes):

        super().__init__()

        self.model = timm.create_model(
            "tf_efficientnetv2_s",
            pretrained=True,
            in_chans=3,
            num_classes=n_classes
        )

    def forward(self, x):

        return self.model(x)

In [14]:
def mixup(x, y, alpha=0.4):

    lam = np.random.beta(alpha, alpha)

    idx = torch.randperm(x.size(0)).to(DEVICE)

    mixed_x = (
        lam * x
        + (1 - lam) * x[idx]
    )

    y_a = y
    y_b = y[idx]

    return mixed_x, y_a, y_b, lam

In [ ]:
skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

fold_scores = []

START_FOLD = 0

for fold, (train_idx, val_idx) in enumerate(

    skf.split(df, df["label_id"])

):

    if fold < START_FOLD:
        continue

    print(f"\nFOLD {fold}")

    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    train_dataset = AudioDataset(
        train_df,
        train=True
    )

    val_dataset = AudioDataset(
        val_df,
        train=False
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    model = AudioModel(
        len(label2id)
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(
        label_smoothing=0.1
    )

    optimizer = optim.AdamW(
        model.parameters(),
        lr=1e-3
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS
    )

    best_f1 = 0

    for epoch in range(EPOCHS):

        model.train()

        train_preds = []
        train_targets = []

        for x, y in tqdm(train_loader):

            x = x.to(DEVICE)
            y = y.to(DEVICE)

            x, y_a, y_b, lam = mixup(x, y)

            optimizer.zero_grad()

            out = model(x)

            loss = (
                lam * criterion(out, y_a)
                + (1 - lam) * criterion(out, y_b)
            )

            loss.backward()

            optimizer.step()

            preds = out.argmax(1)

            train_preds.extend(
                preds.cpu().numpy()
            )

            train_targets.extend(
                y.cpu().numpy()
            )

        model.eval()

        val_preds = []
        val_targets = []

        with torch.no_grad():

            for x, y in val_loader:

                x = x.to(DEVICE)
                y = y.to(DEVICE)

                out = model(x)

                preds = out.argmax(1)

                val_preds.extend(
                    preds.cpu().numpy()
                )

                val_targets.extend(
                    y.cpu().numpy()
                )

        train_f1 = f1_score(
            train_targets,
            train_preds,
            average="weighted"
        )

        val_f1 = f1_score(
            val_targets,
            val_preds,
            average="weighted"
        )

        scheduler.step()

        print(
            f"Epoch {epoch+1} | "
            f"Train F1: {train_f1:.4f} | "
            f"Val F1: {val_f1:.4f}"
        )

        if val_f1 > best_f1:

            best_f1 = val_f1

            torch.save(
                model.state_dict(),
                f"{SAVE_DIR}/fold_{fold}.pt"
            )

    fold_scores.append(best_f1)

    del model
    gc.collect()
    torch.cuda.empty_cache()

print("\nCV:", np.mean(fold_scores))

# Тестирование

In [15]:
test_folder = os.path.join(
    competition_path,
    "audio_test",
    "test"
)

if not os.path.exists(test_folder):

    test_folder = os.path.join(
        competition_path,
        "audio_test"
    )

test_files = sorted(
    os.listdir(test_folder)
)

print("Test files:", len(test_files))

Test files: 3790


In [ ]:
models = []

for fold in range(N_FOLDS):

    path = f"{SAVE_DIR}/fold_{fold}.pt"

    if not os.path.exists(path):
        print("Missing:", path)
        continue

    model = AudioModel(
        len(label2id)
    ).to(DEVICE)

    model.load_state_dict(
        torch.load(
            path,
            map_location=DEVICE
        )
    )

    model.eval()

    models.append(model)

print("Loaded models:", len(models))

In [21]:
def predict_tta_ensemble(path, n=4):

    preds = []

    for _ in range(n):

        x = load_audio(
            path,
            is_train=False
        )

        x = x.unsqueeze(0).to(DEVICE)

        fold_preds = []

        with torch.no_grad(), torch.cuda.amp.autocast():

            for model in models:

                out = torch.softmax(
                    model(x),
                    dim=1
                )

                fold_preds.append(out)

        fold_preds = torch.stack(
            fold_preds
        ).mean(0)

        preds.append(fold_preds)

    preds = torch.stack(preds).mean(0)

    return preds.argmax(1).item()

In [22]:
preds = []
filenames = []

for fname in tqdm(test_files):

    path = os.path.join(
        test_folder,
        fname
    )

    pred = predict_tta_ensemble(
        path,
        n=4
    )

    preds.append(pred)
    filenames.append(fname)

100%|██████████| 3790/3790 [37:29<00:00,  1.68it/s]


In [23]:
pred_labels = [
    id2label[p]
    for p in preds
]

submission = pd.DataFrame({

    "fname": filenames,
    "label": pred_labels
})

submission.to_csv(
    "submission.csv",
    index=False
)

print(submission.head())

                      fname                  label
0  001e64cdd9984e165d34.wav                   Oboe
1  001e949afb7a313b4cfe.wav       Violin_or_fiddle
2  005e20afb8ba6b054afb.wav               Clarinet
3  007c5f37ab13c09bed62.wav              Harmonica
4  008736a59a001197a5d2.wav  Burping_or_eructation
